# 강남구 무더위쉼터 재배치 분석

신규 쉼터를 더하는 대신, **기존 쉼터 일부를 지정 해제하고 같은 수만큼 다른 시설을 새로 지정**하면 접근성이 얼마나 좋아지는지 본다.
무더위쉼터는 건물을 짓는 것이 아니라 기존 시설을 지정하는 제도라서, 쉼터 수를 늘리지 않고도 배치를 바꿀 수 있다.

- 고정 쉼터: 공공청사, 복지·문화·체육시설처럼 해제하지 않는 쉼터. 항상 열려 있다.
- 해제 가능 쉼터: `RELEASABLE_TYPES`에 든 유형(기본값은 회원이용시설, 곧 경로당). 회원이용시설이라도 이름에 `KEEP_NAME_WORDS`(복지관, 센터, 플라자)가 있으면 누구나 쓰는 복지시설이라 고정한다.
- 후보지: 신규 지정을 검토하는 시설. `강남구_입지분석` 노트북과 같은 목록이다.

시나리오는 다섯 가지다. 모두 세 모형(MCLP, 형평성 MCLP, p-median)으로 푼다.

| 시나리오 | 해제할 수 있는 최대 수 | 총 쉼터 수 |
|---|---|---|
| 현재 | 0 | 그대로 |
| 신규 `ADD_COMPARE`곳 추가 (비교 기준) | 0 | + `ADD_COMPARE` |
| 부분 재배치 r곳 (`SWAP_LIST`) | r | 그대로 |
| 전면 재배치 | 해제 가능 쉼터 전부 | 그대로 |

결과는 시나리오별 세 지표, 모형별 교체 목록(해제할 쉼터와 새로 지정할 시설), 여러 모형이 함께 고른 교체 추천 목록, 교체 지도다.

**로컬에서 저장된 결과는 모의 유동인구로 계산한 것이라 해석하면 안 된다.** 안심구역에서는 `0. 설정`의 `SAFE_ZONE`을 `True`로 바꾸고 경로를 확인한 뒤 실행한다.
데이터 행을 그대로 출력하는 셀은 없다. 출력은 건수, 강남구와 행정동 단위 집계, 시설 이름(공개 자료), 그림뿐이다.

1. 자료 불러오기
2. 보행 네트워크와 스냅
3. 고정 쉼터, 해제 가능 쉼터, 후보지
4. 거리 계산과 현재 접근성
5. 재배치 모형
6. 시나리오 풀기
7. 시나리오별 지표
8. 교체 목록과 교체 추천
9. 교체 지도
10. 결과 저장

In [ ]:
import time
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib import font_manager
from matplotlib.lines import Line2D
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import coo_matrix
from scipy.spatial import cKDTree
from shapely import wkt
from shapely.geometry import Point

warnings.filterwarnings("ignore", category=UserWarning)

# 한글 글꼴. 설치된 것 중 먼저 찾은 것을 쓴다.
installed = {f.name for f in font_manager.fontManager.ttflist}
for font_name in ["Malgun Gothic", "NanumGothic", "NanumBarunGothic", "AppleGothic"]:
    if font_name in installed:
        plt.rcParams["font.family"] = font_name
        break
else:
    print("한글 글꼴을 찾지 못했다. 그림의 한글이 깨지면 font_name 목록에 설치된 글꼴 이름을 추가한다.")
plt.rcParams["axes.unicode_minus"] = False

# 색. 서울 EDA와 같은 값이다. 같은 역할에는 같은 색을 쓴다.
INK, INK_SUB, INK_MUTED = "#0b0b0b", "#52514e", "#898781"
SURFACE, LAND, BORDER, GRID = "#fcfcfb", "#f3f2ee", "#c3c2b7", "#e1e0d9"
BLUE = "#2a78d6"     # 기본 계열. 모형 그림에서는 MCLP
ORANGE = "#eb6834"   # 강조. 모형 그림에서는 형평성 MCLP, 공통 입지 지도에서는 신규 선정
AQUA = "#1baf7a"     # 세 번째 계열. 모형 그림에서는 p-median
GRAY = "#b5b3aa"     # 배경 역할
ORANGE_RAMP = ["#f6bfa6", "#eb6834", "#a83c12"]

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": BORDER, "axes.labelcolor": INK_SUB, "text.color": INK,
    "xtick.color": INK_MUTED, "ytick.color": INK_SUB, "axes.titlelocation": "left",
})

## 0. 설정

In [ ]:
SAFE_ZONE = False     # 데이터안심구역에서 실행할 때 True로 바꾼다

if SAFE_ZONE:
    # 안심구역 안의 경로. 실제 파일 이름과 다르면 이 세 줄만 고친다.
    FLOW_PATH = Path("upload/GangnamPopulation7to9.csv")   # 7~9월을 합친 강남구 유동인구
    INPUT = Path("upload")            # 반입한 강남구 경계, 쉼터, 후보시설, 네트워크 파일
    OUT = Path("gangnam_relocation_output")   # 그림과 표가 저장될 폴더
else:
    # 로컬. 유동인구는 강남구만 잘라 둔 모의 자료를 읽는다.
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "2_데이터").exists())
    INPUT = ROOT / "2_데이터" / "강남구_반입"
    FLOW_PATH = INPUT / "GangnamPopulation.csv"
    OUT = ROOT / "4_결과" / "강남구_재배치_모의"

# 대상 지역. 다른 구로 돌릴 때는 이 이름과 아래 네 파일을 그 구의 같은 형식 파일로 바꾼다.
REGION = "강남구"

# 수요 가중치. 안심구역 파일은 weight 열(= population 열)을 그대로 쓴다.
# 이 열이 없는 파일(로컬 모의 자료)은 AGE60_COLS의 남녀 60대 이상 유동인구를 더해 만든다.
WEIGHT_COL = "weight"

BOUNDARY_PATH = INPUT / "GangnamBoundary.shp"         # 행정동 경계
SHELTER_PATH = INPUT / "GangnamCoolingCenter.csv"     # 기존 쉼터
CANDIDATE_PATH = INPUT / "GangnamCandidateFacility.csv"
NETWORK_PATH = INPUT / "GangnamNetwork.csv"           # 경계에서 자른 보행 네트워크

COVERAGE_DISTANCE_M = 219      # 강남구_입지분석과 같은 값. 보행 네트워크 거리
ETA = 0.95                     # 형평성 MCLP가 지킬 MCLP 최적 커버 인구의 비율 (같은 시나리오의 MCLP 기준)
EQUITY_TIEBREAK = True         # 최저 동 커버율이 같은 해가 여럿이면 전체 커버율이 가장 높은 해를 고른다
SNAP_CHECK_M = 100             # 네트워크 노드까지 이보다 멀리 스냅된 점의 수를 보고한다
TIME_LIMIT_S = 300             # 모형 하나를 푸는 시간 상한 (초). 전면 재배치는 변수가 많아 길게 잡았다

# 재배치 설정
RELEASABLE_TYPES = ["회원이용시설"]   # 시설구분2가 이 값인 쉼터만 지정 해제할 수 있다 (경로당)
KEEP_NAME_WORDS = ["복지관", "센터", "플라자"]   # 회원이용시설이라도 이름에 이 말이 있으면 고정한다 (누구나 쓰는 복지시설)
SWAP_LIST = [5, 10, 15]               # 부분 재배치: 해제할 수 있는 최대 곳 수
ADD_COMPARE = 15                      # 비교 기준: 재배치 없이 신규 쉼터만 이만큼 더하는 경우
REC_R = 10                            # 교체 추천 목록과 지도에 쓸 부분 재배치 규모 (SWAP_LIST 중 하나)
KEEP_EPS = 1e-4                       # 효과가 같으면 기존 쉼터를 남기도록 목적함수에 더하는 아주 작은 값의 비율

AGE60_COLS = ["MAN_FLOW_POP_CNT_60GU", "WMAN_FLOW_POP_CNT_60GU"]
MODELS = ["MCLP", "형평성 MCLP", "p-median"]
MODEL_COLOR = {"MCLP": BLUE, "형평성 MCLP": ORANGE, "p-median": AQUA}
MODEL_FILE = {"MCLP": "MCLP", "형평성 MCLP": "형평성MCLP", "p-median": "p-median"}

if REC_R not in SWAP_LIST:
    raise ValueError("REC_R은 SWAP_LIST에 든 값이어야 한다")

SAVE = True
OUT.mkdir(parents=True, exist_ok=True)

# 파일이 제자리에 있는지 먼저 확인한다. 없는 파일이 있으면 여기서 멈춘다.
shp_parts = [BOUNDARY_PATH.with_suffix(s) for s in (".shx", ".dbf", ".prj")]
missing = [str(f) for f in (FLOW_PATH, BOUNDARY_PATH, *shp_parts, SHELTER_PATH, CANDIDATE_PATH, NETWORK_PATH)
           if not f.exists()]
if missing:
    raise FileNotFoundError("다음 파일을 찾을 수 없다: " + ", ".join(missing))
print("입력 파일 확인")

## 0-1. 보조 함수

In [ ]:
def save(fig, name):
    if SAVE:
        fig.savefig(OUT / name, dpi=200, bbox_inches="tight")


def clean(ax, grid="x"):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(axis=grid, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)


def legend_top(ax, handles):
    ax.legend(handles=handles, frameon=False, loc="lower right", bbox_to_anchor=(1, 1.0),
              ncol=len(handles), fontsize=9, labelcolor=INK_SUB, borderaxespad=0.2)


def read_csv_any(path, **kw):
    for enc in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(path, encoding=enc, **kw)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f"{path}를 utf-8, cp949 어느 쪽으로도 읽지 못했다")


def read_points(path):
    df = read_csv_any(path)
    n_raw = len(df)
    df = df.dropna(subset=["경도", "위도"]).reset_index(drop=True)
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["경도"], df["위도"]), crs=4326).to_crs(5179)
    return gdf, n_raw - len(df)


def snap(points, tree, node_ids):
    dist, idx = tree.query(np.column_stack([points.geometry.x, points.geometry.y]))
    out = points.copy()
    out["node"] = node_ids[idx]
    out["snap_m"] = dist
    return out


def weighted_mean(values, weights):
    values, weights = np.asarray(values, float), np.asarray(weights, float)
    ok = np.isfinite(values) & (weights > 0)
    return np.average(values[ok], weights=weights[ok])


def base_map(ax):
    dong.plot(ax=ax, facecolor=LAND, edgecolor=BORDER, linewidth=0.6, zorder=1)
    ax.set_aspect("equal")
    ax.set_axis_off()


def dong_labels(ax):
    for name, pt in zip(dong["dong"], dong.representative_point()):
        ax.text(pt.x, pt.y, name, fontsize=7, color=INK_MUTED, ha="center", va="center", zorder=4.5,
                path_effects=[pe.withStroke(linewidth=2.2, foreground=LAND)])


def context_layers(ax):
    """모든 입지 지도에 공통으로 깔리는 층. 행정동, 후보시설, 기존 쉼터."""
    base_map(ax)
    dong_labels(ax)
    ax.scatter(cand_clean.geometry.x, cand_clean.geometry.y, s=14, facecolors="none",
               edgecolors=GRAY, linewidths=0.8, zorder=4)
    # 기존 쉼터는 회색으로 칠한다. 파랑은 MCLP 선정지 색이라 겹치지 않게 한다.
    ax.scatter(existing_clean.geometry.x, existing_clean.geometry.y, s=22, color=INK_MUTED,
               edgecolors=SURFACE, linewidths=0.8, zorder=5)


def context_handles():
    return [
        Line2D([], [], marker="o", linestyle="", markersize=6, color=INK_MUTED, label=f"기존 쉼터 ({len(existing_clean)}곳)"),
        Line2D([], [], marker="o", linestyle="", markersize=6, markerfacecolor="none",
               markeredgecolor=GRAY, label=f"후보시설 ({len(cand_clean)}곳)"),
    ]


def spread(values, min_gap):
    """끝 라벨이 겹치지 않게 y 위치를 min_gap 이상 벌린다. 순서는 유지한다."""
    order = np.argsort(values)
    pos = np.asarray(values, float)[order].copy()
    for _ in range(50):
        for k in range(1, len(pos)):
            if pos[k] - pos[k - 1] < min_gap:
                mid = (pos[k] + pos[k - 1]) / 2
                pos[k - 1], pos[k] = mid - min_gap / 2, mid + min_gap / 2
    out = np.empty_like(pos)
    out[order] = pos
    return out

## 1. 자료 불러오기

60대 이상 유동인구를 수요로 쓴다. 안심구역 파일은 `weight` 열을, 로컬 모의 파일은 남녀 60대 이상 열의 합을 쓴다.
격자에 행정동을 붙이고 강남구 밖 격자는 버린다. 파일에 행정동(`ADM_NM`)이 이미 있으면 경계로 다시 붙인 결과와 얼마나 일치하는지 출력한다.
좌표는 `X_COORD`, `Y_COORD`(EPSG:5179), `lon`, `lat`, `geometry`(WKT) 순서로 있는 것을 쓴다. 구분자(`,`, `|`)와 인코딩은 자동으로 판단한다.
기존 쉼터는 강남구 98곳만 쓴다. 경계 밖 쉼터는 넣지 않았으므로 경계부 수요지의 커버율은 실제보다 낮게 나올 수 있다.

In [ ]:
dong = gpd.read_file(BOUNDARY_PATH).to_crs(5179).rename(columns={"ADM_NM": "dong"})[["dong", "geometry"]]

with open(FLOW_PATH, "rb") as f:
    header = f.readline().decode("utf-8", errors="ignore")
flow = read_csv_any(FLOW_PATH, sep="|" if header.count("|") > header.count(",") else ",", low_memory=False)
cols = set(flow.columns)

# 이전 공간결합에서 남은 열. 그대로 두면 아래 sjoin이 열 이름 충돌로 멈춘다.
flow = flow.drop(columns=[c for c in ("index_right", "index_left") if c in cols])

if WEIGHT_COL in cols:
    flow["weight"] = pd.to_numeric(flow[WEIGHT_COL], errors="coerce").fillna(0)
    weight_source = f"{WEIGHT_COL} 열"
elif set(AGE60_COLS) <= cols:
    flow["weight"] = flow[AGE60_COLS].apply(pd.to_numeric, errors="coerce").sum(axis=1)
    weight_source = " + ".join(AGE60_COLS)
else:
    raise KeyError(f"가중치 열을 찾지 못했다. 파일의 열: {sorted(cols)}")

if {"X_COORD", "Y_COORD"} <= cols:
    pts = gpd.points_from_xy(pd.to_numeric(flow["X_COORD"]), pd.to_numeric(flow["Y_COORD"]))
    flow = gpd.GeoDataFrame(flow.drop(columns="geometry", errors="ignore"), geometry=pts, crs=5179)  # 문자열 geometry 열은 버린다
    coord_source = "X_COORD, Y_COORD (EPSG:5179)"
elif {"lon", "lat"} <= cols:
    pts = gpd.points_from_xy(pd.to_numeric(flow["lon"]), pd.to_numeric(flow["lat"]))
    flow = gpd.GeoDataFrame(flow.drop(columns="geometry", errors="ignore"), geometry=pts, crs=4326).to_crs(5179)
    coord_source = "lon, lat (EPSG:4326)"
elif "geometry" in cols:
    geom = gpd.GeoSeries.from_wkt(flow["geometry"].astype(str))
    crs = 4326 if abs(geom.x).max() <= 180 else 5179       # 경위도인지 미터 좌표인지 값 크기로 판단한다
    flow = gpd.GeoDataFrame(flow.drop(columns="geometry"), geometry=geom.values, crs=crs).to_crs(5179)
    coord_source = f"geometry WKT (EPSG:{crs})"
else:
    raise KeyError(f"좌표 열을 찾지 못했다. 파일의 열: {sorted(cols)}")

if "STD_YM" in cols:
    print("STD_YM 값:", sorted(flow["STD_YM"].unique()), "(여러 달이면 모두 더해진다)")

n_flow = len(flow)
flow = gpd.sjoin(flow, dong, how="inner", predicate="within").drop(columns="index_right")
if "ADM_NM" in flow.columns:
    agree = (flow["ADM_NM"].astype(str) == flow["dong"]).mean() * 100
    print(f"파일의 ADM_NM과 경계로 붙인 행정동 일치: {agree:.1f}%")
flow = flow[flow["weight"] > 0].reset_index(drop=True)

existing, n_shelter_drop = read_points(SHELTER_PATH)
cand, n_cand_drop = read_points(CANDIDATE_PATH)

print("유동인구 파일:", FLOW_PATH.name)
print("가중치:", weight_source, "| 좌표:", coord_source)
print("행정동:", f"{len(dong)}개")
print("유동인구 격자:", f"{n_flow:,}개 중 강남구 안, 가중치 0 초과 {len(flow):,}개")
print("기존 쉼터:", f"{len(existing)}곳 (좌표 없음 {n_shelter_drop}곳)")
print("후보시설:", f"{len(cand)}곳 (좌표 없음 {n_cand_drop}곳)")

## 2. 보행 네트워크와 스냅

링크 길이를 가중치로 그래프를 만들고 가장 큰 연결요소만 쓴다.
네트워크가 강남구 경계에서 잘려 있어서 잘린 링크의 끝점에는 NODE 행이 없다. 그 좌표는 링크 WKT의 첫 점과 끝 점으로 채운다.
수요 격자, 기존 쉼터, 후보시설은 가장 가까운 노드에 붙인다.

In [ ]:
net = read_csv_any(NETWORK_PATH, dtype=str, low_memory=False)

links = net[net["노드링크 유형"] == "LINK"].copy()
links["length"] = pd.to_numeric(links["링크 길이"], errors="coerce")
links = links[np.isfinite(links["length"]) & (links["length"] > 0)]

node_rows = net[(net["노드링크 유형"] == "NODE") & net["노드 WKT"].notna()]
node_pt = dict(zip(node_rows["노드 ID"], node_rows["노드 WKT"].map(wkt.loads)))

n_recovered = 0
for s, e, g in zip(links["시작노드 ID"], links["종료노드 ID"], links["링크 WKT"]):
    if (s in node_pt and e in node_pt) or not isinstance(g, str):
        continue
    line = wkt.loads(g)
    parts = list(line.geoms) if line.geom_type == "MultiLineString" else [line]
    if s not in node_pt:
        node_pt[s] = Point(parts[0].coords[0])
        n_recovered += 1
    if e not in node_pt:
        node_pt[e] = Point(parts[-1].coords[-1])
        n_recovered += 1

G = nx.Graph()
for u, v, length in zip(links["시작노드 ID"], links["종료노드 ID"], links["length"]):
    if u == v:
        continue
    if not G.has_edge(u, v) or length < G[u][v]["length"]:
        G.add_edge(u, v, length=float(length))
n_graph_all = G.number_of_nodes()
G = G.subgraph(max(nx.connected_components(G), key=len)).copy()

node_ids = np.array([n for n in G.nodes if n in node_pt])
node_geo = gpd.GeoSeries([node_pt[n] for n in node_ids], crs=4326).to_crs(5179)
tree = cKDTree(np.column_stack([node_geo.x, node_geo.y]))

flow = snap(flow, tree, node_ids)
existing = snap(existing, tree, node_ids)
cand = snap(cand, tree, node_ids)

print("링크:", f"{len(links):,}개, 좌표를 복원한 노드 {n_recovered}개")
print("그래프 노드:", f"{n_graph_all:,}개 중 가장 큰 연결요소 {G.number_of_nodes():,}개 "
      f"({G.number_of_nodes() / n_graph_all * 100:.1f}%)")
for label, pts in [("수요 격자", flow), ("기존 쉼터", existing), ("후보시설", cand)]:
    print(f"{label} 스냅 거리: 중앙값 {pts['snap_m'].median():.1f}m, 최대 {pts['snap_m'].max():.0f}m, "
          f"{SNAP_CHECK_M}m 초과 {(pts['snap_m'] > SNAP_CHECK_M).sum()}개")

## 2-1. 분석용 수요지, 기존 쉼터, 후보지

- 같은 노드에 붙은 수요 격자는 가중치를 더한다. 행정동이 다르면 따로 둔다.
- 같은 노드에 붙은 후보시설은 하나만 남긴다.
- 기존 쉼터와 같은 노드에 붙은 후보시설은 뺀다.

In [ ]:
existing_clean = existing.drop_duplicates("node").copy()
existing_nodes = existing_clean["node"].tolist()

cand_clean = (cand.sort_values("snap_m").drop_duplicates("node")
              .loc[lambda d: ~d["node"].isin(existing_nodes)].sort_index().reset_index(drop=True))
cand_clean["candidate_id"] = [f"C{i:03d}" for i in range(1, len(cand_clean) + 1)]
cand_clean["label"] = cand_clean["candidate_id"] + " " + cand_clean["시설명"].astype(str)

demand = flow.groupby(["node", "dong"], as_index=False).agg(weight=("weight", "sum"), n_grid=("weight", "size"))
w = demand["weight"].to_numpy(float)
dong_of = demand["dong"].to_numpy()
pop_dong = pd.Series(w).groupby(dong_of).sum()
n_cand = len(cand_clean)

print("기존 쉼터:", f"{len(existing)}곳 → 노드 기준 {len(existing_clean)}곳")
print("후보지:", f"{len(cand)}곳 → 중복 노드, 기존 쉼터 노드 제외 후 {n_cand}곳")
print("수요지:", f"격자 {len(flow):,}개 → 노드 기준 {len(demand):,}개, 행정동 {demand['dong'].nunique()}개")

## 3. 고정 쉼터, 해제 가능 쉼터, 후보지

한 노드에 쉼터가 여럿 붙어 있고 그중 하나라도 고정 유형이면 그 노드는 고정으로 본다.
해제 가능 쉼터와 후보지를 합친 목록(`mov`)이 모형이 고르는 대상이다. 앞쪽 `n_r`개가 기존 쉼터, 뒤쪽이 후보지다.

In [ ]:
type_col = "시설구분2"
names = existing["쉼터명칭"].astype(str)
releasable_row = (existing[type_col].astype(str).isin(RELEASABLE_TYPES)
                  & ~names.str.contains("|".join(KEEP_NAME_WORDS)))
kept_by_name = existing[existing[type_col].astype(str).isin(RELEASABLE_TYPES) & ~releasable_row]
fixed_node_set = set(existing.loc[~releasable_row, "node"])
existing_clean["releasable"] = ~existing_clean["node"].isin(fixed_node_set)
fixed = existing_clean[~existing_clean["releasable"]].reset_index(drop=True)
rel = existing_clean[existing_clean["releasable"]].reset_index(drop=True)
if len(fixed) == 0:
    raise ValueError("고정 쉼터가 없다. p-median은 항상 열려 있는 쉼터가 있어야 배정 쌍을 줄일 수 있다. RELEASABLE_TYPES를 확인한다.")

n_r, n_c = len(rel), len(cand_clean)
n_m = n_r + n_c
R_IDX, C_IDX = np.arange(n_r), np.arange(n_r, n_m)

mov = gpd.GeoDataFrame({
    "구분": ["기존"] * n_r + ["후보"] * n_c,
    "id": [f"E{i:03d}" for i in range(1, n_r + 1)] + cand_clean["candidate_id"].tolist(),
    "시설명": rel["쉼터명칭"].astype(str).tolist() + cand_clean["시설명"].astype(str).tolist(),
    "유형": rel[type_col].astype(str).tolist() + cand_clean["시설유형"].astype(str).tolist(),
    "node": rel["node"].tolist() + cand_clean["node"].tolist(),
}, geometry=list(rel.geometry) + list(cand_clean.geometry), crs=5179)

print("기존 쉼터 유형:", existing_clean[type_col].astype(str).value_counts().to_dict())
print("회원이용시설이지만 이름 때문에 고정한 쉼터:", ", ".join(kept_by_name["쉼터명칭"].astype(str)))
print(f"고정 쉼터 {len(fixed)}곳, 해제 가능 쉼터 {n_r}곳, 후보지 {n_c}곳")

## 4. 거리 계산과 현재 접근성

고정 쉼터까지 거리(`ef`)는 다중 출발점 다익스트라로 한 번에 구한다. 고정 쉼터는 항상 열려 있으므로 어느 해에서도 수요지의 거리는 `ef`보다 길어지지 않는다.
그래서 해제 가능 쉼터와 후보지까지 거리(`D`)는 `ef`의 최댓값에서 끊어도 결과가 같다.
현재 상태(해제 가능 쉼터를 모두 연 상태)의 지표는 `강남구_입지분석` 노트북의 기존 쉼터 지표와 같아야 한다.

In [ ]:
near_fixed = nx.multi_source_dijkstra_path_length(G, fixed["node"].tolist(), weight="length")
ef = np.array([near_fixed.get(n, np.inf) for n in demand["node"]])
if not np.isfinite(ef).all():
    raise RuntimeError("고정 쉼터에 닿지 않는 수요지가 있다. 네트워크 연결이나 RELEASABLE_TYPES를 확인한다.")
cutoff = max(COVERAGE_DISTANCE_M, ef.max())

rows_by_node = defaultdict(list)
for i, n in enumerate(demand["node"]):
    rows_by_node[n].append(i)

D = np.full((len(demand), n_m), np.inf)
for j, src in enumerate(mov["node"]):
    for n, d in nx.single_source_dijkstra_path_length(G, src, cutoff=cutoff, weight="length").items():
        for i in rows_by_node.get(n, ()):
            D[i, j] = d


def evaluate(open_idx):
    """열린 mov 목록을 받아 세 지표를 계산한다. 고정 쉼터는 항상 포함한다."""
    open_idx = np.asarray(open_idx, int)
    dist = ef if len(open_idx) == 0 else np.minimum(ef, D[:, open_idx].min(axis=1))
    covered = dist <= COVERAGE_DISTANCE_M
    by_dong = pd.Series(w * covered).groupby(dong_of).sum() / pop_dong
    return {
        "coverage_pct": w[covered].sum() / w.sum() * 100,
        "min_dong_pct": by_dong.min() * 100,
        "mean_dist_m": weighted_mean(dist, w),
        "covered_people": w[covered].sum(),
    }


now = evaluate(R_IDX)
only_fixed = evaluate([])
print(f"기준거리 {COVERAGE_DISTANCE_M}m, 보행 네트워크 거리")
print(f"현재 커버율 {now['coverage_pct']:.1f}%, 최저 행정동 {now['min_dong_pct']:.1f}%, 평균 보행거리 {now['mean_dist_m']:.0f}m")
print(f"참고: 해제 가능 쉼터를 모두 닫으면 커버율 {only_fixed['coverage_pct']:.1f}%")

## 5. 재배치 모형

변수 `x`는 `mov`의 각 시설을 쉼터로 둘지(1) 말지(0)다. 기존 쉼터 자리에서 `x = 0`이면 지정 해제, 후보지 자리에서 `x = 1`이면 신규 지정이다.
세 모형 모두 아래 두 제약을 공통으로 쓴다.

- 총 쉼터 수: `Σ x = n_r + p_add` (재배치 시나리오는 `p_add = 0`이라 쉼터 수가 그대로다)
- 해제 상한: `Σ_{기존} x ≥ n_r − r` (기존 쉼터를 r곳까지만 닫을 수 있다)

목적함수와 나머지 제약은 `강남구_입지분석`과 같다. 계산 방식도 같게 두었다.

- MCLP: 고정 쉼터가 이미 커버하는 수요지는 상수로 빼고, 나머지 수요지를 커버 시설 집합이 같은 것끼리 묶는다. `y ≤ Σ x`.
- 형평성 MCLP: 행정동별 커버율의 최솟값 q를 최대화한다. 전체 커버 인구는 같은 시나리오 MCLP 최적값의 `ETA` 이상.
- p-median: 각 수요지를 고정 쉼터(거리 `ef`) 또는 그보다 가까운 열린 시설 하나에 배정한다.

효과가 같은 해가 여럿이면 쓸데없는 교체가 목록에 섞인다. 그래서 MCLP, p-median, 형평성 MCLP 2단계 목적함수에 기존 쉼터를 남길수록 아주 조금 유리한 항(`KEEP_EPS`)을 더했다. 수요 가중치 최솟값의 1만분의 1 크기라서 최적 커버 인구나 거리는 바뀌지 않는다.

In [ ]:
cover = np.isfinite(D) & (D <= COVERAGE_DISTANCE_M)
base_cov = ef <= COVERAGE_DISTANCE_M                       # 고정 쉼터만으로 커버되는 수요지
base_people = w[base_cov].sum()
open_rows = np.flatnonzero(~base_cov & cover.any(axis=1))   # 열리는 시설에 따라 커버 여부가 바뀌는 수요지
eps = KEEP_EPS * w[w > 0].min()


def group_rows(rows, by_dong=False):
    keys = [r.tobytes() for r in np.packbits(cover[rows], axis=1)]
    df = pd.DataFrame({"key": keys, "row": rows, "w": w[rows], "dong": dong_of[rows]})
    g = (df.groupby(["key", "dong"] if by_dong else ["key"], sort=False)
         .agg(w=("w", "sum"), first=("row", "first"), dong=("dong", "first")).reset_index(drop=True))
    g["cands"] = [np.flatnonzero(cover[i]) for i in g["first"]]
    return g


def link_matrix(groups, n_vars):
    """y_g - (g를 커버하는 시설 x의 합) <= 0"""
    m = len(groups)
    r = np.concatenate([np.full(len(c), k) for k, c in enumerate(groups["cands"])])
    c = np.concatenate(groups["cands"].to_list())
    return coo_matrix((np.r_[-np.ones(len(r)), np.ones(m)], (np.r_[r, np.arange(m)], np.r_[c, n_m + np.arange(m)])),
                      shape=(m, n_vars)).tocsr()


def run_milp(c, constraints, n_vars):
    integrality = np.r_[np.ones(n_m), np.zeros(n_vars - n_m)]
    res = milp(c, constraints=constraints, integrality=integrality, bounds=Bounds(0, 1),
               options={"time_limit": TIME_LIMIT_S, "mip_rel_gap": 1e-6})
    if res.x is None:
        raise RuntimeError(f"해를 찾지 못했다: {res.message}")
    if res.status != 0:
        print(f"  주의: {res.message}")
    return res


def scenario_cons(n_vars, r, p_add):
    total = np.zeros(n_vars)
    total[:n_m] = 1
    keep = np.zeros(n_vars)
    keep[R_IDX] = 1
    return [LinearConstraint(total[None, :], n_r + p_add, n_r + p_add),   # 총 쉼터 수
            LinearConstraint(keep[None, :], n_r - r, n_r)]                 # 해제 상한


def keep_bonus(n_vars):
    """최소화 목적함수에 더한다. 기존 쉼터를 남길수록 아주 조금 작아진다."""
    v = np.zeros(n_vars)
    v[R_IDX] = -eps
    return v


def chosen(res):
    return np.flatnonzero(res.x[:n_m] > 0.5)


# MCLP. 변수 순서는 x, y
grp_m = group_rows(open_rows)
nv_m = n_m + len(grp_m)
link_m = link_matrix(grp_m, nv_m)
c_m = np.r_[np.zeros(n_m), -grp_m["w"].to_numpy()] + keep_bonus(nv_m)


def solve_mclp(r, p_add):
    res = run_milp(c_m, [LinearConstraint(link_m, -np.inf, 0), *scenario_cons(nv_m, r, p_add)], nv_m)
    return chosen(res)


# 형평성 MCLP. 변수 순서는 x, y, q
grp_e = group_rows(open_rows, by_dong=True)
nv_e = n_m + len(grp_e) + 1
link_e = link_matrix(grp_e, nv_e)
dongs = pop_dong.index
base_dong = pd.Series(w * base_cov).groupby(dong_of).sum().reindex(dongs).fillna(0).to_numpy()
d_idx = dongs.get_indexer(grp_e["dong"])
dong_A = coo_matrix(
    (np.r_[grp_e["w"].to_numpy(), -pop_dong.to_numpy()],
     (np.r_[d_idx, np.arange(len(dongs))], np.r_[n_m + np.arange(len(grp_e)), np.full(len(dongs), nv_e - 1)])),
    shape=(len(dongs), nv_e)).tocsr()
total_e = np.r_[np.zeros(n_m), grp_e["w"].to_numpy(), 0.0]
q_row = np.zeros(nv_e)
q_row[-1] = 1


def solve_equity(r, p_add, mclp_people):
    cons = [LinearConstraint(link_e, -np.inf, 0),
            *scenario_cons(nv_e, r, p_add),
            LinearConstraint(dong_A, -base_dong, np.inf),                          # 동별 커버 인구 >= q x 동 인구
            LinearConstraint(total_e[None, :], ETA * mclp_people - base_people, np.inf)]
    res = run_milp(-q_row, cons, nv_e)
    if EQUITY_TIEBREAK:
        cons.append(LinearConstraint(q_row[None, :], res.x[-1] - 1e-7, np.inf))
        res = run_milp(-total_e + keep_bonus(nv_e), cons, nv_e)
    return chosen(res)


# p-median. 고정 쉼터보다 가까운 시설만 배정 대상으로 둔다. 변수 순서는 x, u, v
pi, pj = np.nonzero(np.isfinite(D) & (D < ef[:, None]))
rows_pm = np.unique(pi)
ui = np.searchsorted(rows_pm, pi)
nu, nk = len(rows_pm), len(pi)
nv_p = n_m + nu + nk
c_p = np.r_[np.zeros(n_m), w[rows_pm] * ef[rows_pm], w[pi] * D[pi, pj]] + keep_bonus(nv_p)
assign = coo_matrix((np.ones(nu + nk), (np.r_[np.arange(nu), ui], np.r_[n_m + np.arange(nu), n_m + nu + np.arange(nk)])),
                    shape=(nu, nv_p)).tocsr()                                      # u_i + sum_k v_k = 1
opened = coo_matrix((np.r_[np.ones(nk), -np.ones(nk)], (np.r_[np.arange(nk), np.arange(nk)], np.r_[n_m + nu + np.arange(nk), pj])),
                    shape=(nk, nv_p)).tocsr()                                      # v_k <= x_j


def solve_pmedian(r, p_add):
    cons = [LinearConstraint(assign, 1, 1), LinearConstraint(opened, -np.inf, 0), *scenario_cons(nv_p, r, p_add)]
    res = run_milp(c_p, cons, nv_p)
    return chosen(res)


print(f"MCLP: 묶은 수요지 {len(grp_m):,}개 (원래 {len(open_rows):,}개)")
print(f"형평성 MCLP: 묶은 수요지 {len(grp_e):,}개")
print(f"p-median: 배정 쌍 {nk:,}개, 대상 수요지 {nu:,}개")

## 6. 시나리오 풀기

시나리오마다 MCLP를 먼저 풀고, 그 최적 커버 인구로 형평성 MCLP의 하한을 정한다.
`해제`는 지정 해제한 기존 쉼터 수, `신규`는 새로 지정한 시설 수다. 재배치 시나리오에서는 둘이 같다.

In [ ]:
SCENARIOS = ([("현재", 0, 0), (f"신규 {ADD_COMPARE}곳 추가", 0, ADD_COMPARE)]
             + [(f"재배치 {r}곳", r, 0) for r in SWAP_LIST]
             + [("전면 재배치", n_r, 0)])
SCEN_NAMES = [s[0] for s in SCENARIOS]

records, solutions = [], {}
t0 = time.time()
for name, r, p_add in SCENARIOS:
    if name == "현재":
        sols = {m: R_IDX for m in MODELS}
    else:
        sel_m = solve_mclp(r, p_add)
        sols = {"MCLP": sel_m,
                "형평성 MCLP": solve_equity(r, p_add, evaluate(sel_m)["covered_people"]),
                "p-median": solve_pmedian(r, p_add)}
    for model, sel in sols.items():
        if len(sel) != n_r + p_add:
            raise RuntimeError(f"{name} {model}: 쉼터 수가 {len(sel)}곳이다")
        solutions[(name, model)] = sel
        records.append({"scenario": name, "model": model,
                        "해제": n_r - np.isin(R_IDX, sel).sum(), "신규": np.isin(C_IDX, sel).sum(),
                        **evaluate(sel)})
    print(f"{name} 완료 ({time.time() - t0:.0f}초)")

metrics = pd.DataFrame(records)

## 7. 시나리오별 지표

세 지표를 시나리오별로 비교한다. 같은 모형 안에서 "신규 추가"와 "재배치"를 나란히 보면, 쉼터 수를 늘리지 않고 위치만 바꿔서 얻는 효과가 신규 추가와 비교해 어느 정도인지 알 수 있다.

In [ ]:
table = (metrics.assign(scenario=pd.Categorical(metrics["scenario"], SCEN_NAMES, ordered=True))
         .pivot_table(index="scenario", columns="model", values=["coverage_pct", "min_dong_pct", "mean_dist_m", "해제"],
                      observed=False)
         .round(1))
display(table)

panels = [("coverage_pct", "커버율 (%)"), ("min_dong_pct", "최저 행정동 커버율 (%)"), ("mean_dist_m", "평균 보행거리 (m)")]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
xs = np.arange(len(SCEN_NAMES))
bw = 0.26
for ax, (col, label) in zip(axes, panels):
    for k, model in enumerate(MODELS):
        s = metrics[metrics["model"] == model].set_index("scenario").loc[SCEN_NAMES, col]
        ax.bar(xs + (k - 1) * bw, s.to_numpy(), width=bw * 0.92, color=MODEL_COLOR[model], label=model)
    ax.axhline(now[col], color=INK_SUB, linewidth=1, linestyle=(0, (4, 3)))
    ax.set_xticks(xs)
    ax.set_xticklabels(SCEN_NAMES, rotation=30, ha="right", fontsize=9)
    ax.set_ylabel(label)
    clean(ax, "y")
legend_top(axes[0], [Line2D([], [], marker="s", linestyle="", markersize=9, color=MODEL_COLOR[m], label=m) for m in MODELS])
fig.tight_layout()
save(fig, "01_시나리오별_지표.png")
plt.show()

## 8. 교체 목록과 교체 추천

모형마다 어떤 쉼터를 해제하고 어떤 시설을 새로 지정했는지 정리한다.
`REC_R` 규모에서 세 모형 중 몇 개가 같은 교체를 골랐는지 세어, 여러 모형이 함께 고른 교체를 추천 목록으로 둔다.
모형 수가 많을수록 목적을 바꿔도 남는 교체라서 우선 검토할 만하다.

In [ ]:
swap_rows = []
for (name, model), sel in solutions.items():
    if name == "현재":
        continue
    for j in R_IDX[~np.isin(R_IDX, sel)]:
        swap_rows.append({"scenario": name, "model": model, "변경": "해제", **mov.loc[j, ["id", "시설명", "유형"]].to_dict()})
    for j in C_IDX[np.isin(C_IDX, sel)]:
        swap_rows.append({"scenario": name, "model": model, "변경": "신규 지정", **mov.loc[j, ["id", "시설명", "유형"]].to_dict()})
swaps = pd.DataFrame(swap_rows, columns=["scenario", "model", "변경", "id", "시설명", "유형"])

rec_name = f"재배치 {REC_R}곳"
rec = swaps[swaps["scenario"] == rec_name]
recommend = (rec.groupby(["변경", "id", "시설명", "유형"])["model"]
             .agg(모형수="nunique", 모형=lambda s: ", ".join(sorted(s))).reset_index()
             .sort_values(["변경", "모형수", "id"], ascending=[True, False, True]))
print(f"{rec_name}: 모형별 해제 수", rec[rec["변경"] == "해제"].groupby("model").size().to_dict())
display(recommend.set_index("id"))

## 9. 교체 지도

`REC_R` 규모에서 모형마다 해제한 쉼터(검은 X)와 새로 지정한 시설(모형 색)을 그린다. 고정 쉼터와 남긴 쉼터는 회색이다.
마지막 장은 전면 재배치 MCLP의 결과다.

In [ ]:
def relocation_map(name, model):
    sel = solutions[(name, model)]
    closed = mov.loc[R_IDX[~np.isin(R_IDX, sel)]]
    kept = mov.loc[R_IDX[np.isin(R_IDX, sel)]]
    new = mov.loc[C_IDX[np.isin(C_IDX, sel)]]
    fig, ax = plt.subplots(figsize=(8.5, 8.5))
    base_map(ax)
    dong_labels(ax)
    ax.scatter(cand_clean.geometry.x, cand_clean.geometry.y, s=14, facecolors="none", edgecolors=GRAY, linewidths=0.8, zorder=4)
    ax.scatter(fixed.geometry.x, fixed.geometry.y, s=22, color=INK_MUTED, edgecolors=SURFACE, linewidths=0.8, zorder=5)
    ax.scatter(kept.geometry.x, kept.geometry.y, s=22, color=INK_MUTED, edgecolors=SURFACE, linewidths=0.8, zorder=5)
    ax.scatter(closed.geometry.x, closed.geometry.y, s=70, marker="x", color=INK, linewidths=2, zorder=6)
    ax.scatter(new.geometry.x, new.geometry.y, s=75, color=MODEL_COLOR[model], edgecolors=SURFACE, linewidths=1.5, zorder=7)
    handles = [
        Line2D([], [], marker="o", linestyle="", markersize=6, color=INK_MUTED, label=f"유지 쉼터 ({len(fixed) + len(kept)}곳)"),
        Line2D([], [], marker="x", linestyle="", markersize=8, color=INK, markeredgewidth=2, label=f"지정 해제 ({len(closed)}곳)"),
        Line2D([], [], marker="o", linestyle="", markersize=8, color=MODEL_COLOR[model], label=f"신규 지정 ({len(new)}곳)"),
        Line2D([], [], marker="o", linestyle="", markersize=6, markerfacecolor="none", markeredgecolor=GRAY, label=f"후보시설 ({n_c}곳)"),
    ]
    ax.legend(handles=handles, title=f"{model}, {name}, 기준거리 {COVERAGE_DISTANCE_M}m", frameon=False,
              loc="lower left", fontsize=9, title_fontsize=9, labelcolor=INK_SUB)
    return fig


for model in MODELS:
    fig = relocation_map(rec_name, model)
    save(fig, f"02_재배치지도_{MODEL_FILE[model]}_r{REC_R}.png")
    plt.show()

fig = relocation_map("전면 재배치", "MCLP")
save(fig, "03_전면재배치_MCLP.png")
plt.show()

## 10. 결과 저장

저장하는 표는 시나리오별 지표(강남구, 행정동 집계)와 시설 이름(공개 자료)뿐이다. 격자 단위 값은 저장하지 않는다.

In [ ]:
if SAVE:
    metrics.drop(columns="covered_people").to_csv(OUT / "재배치_시나리오별_지표.csv", index=False, encoding="utf-8-sig")
    swaps.to_csv(OUT / "재배치_교체목록.csv", index=False, encoding="utf-8-sig")
    recommend.to_csv(OUT / f"재배치_교체추천_r{REC_R}.csv", index=False, encoding="utf-8-sig")
    print("저장 위치:", OUT)

## 실제 자료로 돌릴 때

- `SAFE_ZONE = True`로 바꾸고 `FLOW_PATH`, `INPUT`을 확인한다. 반입 파일은 `강남구_입지분석_안심구역`과 같다(경계 다섯 개, 쉼터, 후보시설, 네트워크).
- 해제 대상을 바꾸려면 `RELEASABLE_TYPES`만 고친다. 예를 들어 복지·문화·체육시설도 해제 대상으로 두려면 그 값을 목록에 더한다.
- 다른 구로 돌리려면 `REGION`과 네 입력 파일을 그 구의 같은 형식 파일로 바꾼다. 열 이름(쉼터명칭, 시설구분2, 시설명, 시설유형, 위도, 경도, ADM_NM)은 같아야 한다.
- 반출할 것은 `OUT` 폴더의 CSV 세 개와 PNG 다섯 개다. 격자 단위 값은 들어 있지 않다.